# Bayesian Optimization — NASA Battery B0005

Bayesian search over the two architectures of the paper, then a benchmark table placing the
results next to the published works.

1. **Sequential CNN-LSTM** — 30 trials
2. **Parallel CNN-LSTM** — 30 trials
3. **Benchmark table** — our results against Tian, Gao and Khan

Run from the folder containing `B0005.mat`. Roughly 45-60 minutes; Keras Tuner checkpoints each
trial, so an interrupted run resumes.

Two notes, both small but they affect whether the numbers hold up:

- Each trial is scored as the mean of 3 seeds. On this dataset one architecture re-run with a
  different seed varies by 5-10x, so a single-seed score would let the search pick a lucky run
  and return a configuration that does not reproduce.
- Results are produced on **both** the one-step protocol (which your current 0.047 Ah figure
  uses) and the **recursive** protocol (which Tian and Gao use). The benchmark table compares
  like with like on the recursive rows.

In [ ]:
import os, json, gc, time
os.environ['PYTHONHASHSEED'] = '42'
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'
import numpy as np, pandas as pd
from sklearn.metrics import mean_squared_error
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.layers import (Input, Conv1D, MaxPooling1D, Dropout, Dense, LSTM,
                                     Concatenate, GlobalMaxPooling1D, GlobalAveragePooling1D)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import keras_tuner as kt
try: tf.config.experimental.enable_op_determinism()
except Exception as e: print('op determinism unavailable:', e)
print('TF', tf.__version__, '| keras-tuner', kt.__version__,
      '| GPU', tf.config.list_physical_devices('GPU'))

In [ ]:
MAT_PATH   = 'B0005.mat'
WINDOW     = 10
SPLIT_ONESTEP, SPLIT_RECUR = 50, 80     # 80 matches Tian SP=80% and Gao start point 80
MAX_TRIALS = 30
SEEDS_TRIAL, SEEDS_FINAL = 3, 10
SEARCH_EPOCHS, SEARCH_PATIENCE = 150, 20
FINAL_EPOCHS,  FINAL_PATIENCE  = 300, 30
BATCH, SEED = 8, 42
np.random.seed(SEED); tf.random.set_seed(SEED)

from scipy.io import loadmat
mat = loadmat(MAT_PATH); cycles = mat['B0005'][0, 0]['cycle'][0]
rows, n = [], 0
for i in range(len(cycles)):
    r = cycles[i]
    if r['type'][0] != 'discharge': continue
    n += 1; rows.append(dict(cycle=n, capacity=float(r['data'][0][0]['Capacity'][0][0])))
HI = pd.DataFrame(rows); C0 = HI['capacity'].iloc[0]
HI['SOH'] = (HI['capacity'] / C0).astype(np.float32)
S_ALL, CYC = HI['SOH'].values.astype(np.float32), HI['cycle'].values
print('cycles %d | initial capacity %.4f Ah' % (len(HI), C0))

def make_split(sc):
    Xtr, ytr, ptr, Xte, yte, pte = [], [], [], [], [], []
    for e in range(WINDOW, len(HI)):
        seq = S_ALL[e-WINDOW:e].reshape(WINDOW, 1); yv = S_ALL[e] - S_ALL[e-1]
        if CYC[e] < sc: Xtr.append(seq); ytr.append(yv); ptr.append(S_ALL[e-1])
        else:           Xte.append(seq); yte.append(yv); pte.append(S_ALL[e-1])
    f = lambda a: np.asarray(a, np.float32); Xtr, ytr, ptr = f(Xtr), f(ytr), f(ptr)
    cut = int(0.75*len(Xtr))
    return dict(Xf=Xtr[:cut], yf=ytr[:cut], Xv=Xtr[cut:], yv=ytr[cut:], pv=ptr[cut:],
                Xt=f(Xte), yt=f(yte), pt=f(pte),
                val_start=WINDOW+cut, test_start=int(np.where(CYC >= sc)[0][0]))
DATA = {sc: make_split(sc) for sc in (SPLIT_ONESTEP, SPLIT_RECUR)}
for sc, d in DATA.items():
    print('split %d | fit %s val %s test %s' % (sc, d['Xf'].shape, d['Xv'].shape, d['Xt'].shape))

## Search space

Mirrors the N-CMAPSS Bayesian notebook: CNN depth 1-4 with filters, kernel size, dropout and
pooling; LSTM depth 1-3 with units and dropouts; 1-3 dense layers; optimizer and learning rate.
The only change is that filters and units start at 8 rather than 32, because B0005 gives ~30-50
training windows and a floor of 32 units forces every trial to over-fit. Upper bounds are
unchanged, so this space contains the N-CMAPSS one.

In [ ]:
FILT = dict(min_value=8, max_value=128, step=8)
UNIT = dict(min_value=8, max_value=128, step=8)
GI   = lambda s: tf.keras.initializers.GlorotUniform(seed=s)

def _cnn(x, hp, s):
    for i in range(hp.Int('num_cnn_layers', 1, 4)):
        x = Conv1D(hp.Int('cnn_%d_filters' % i, **FILT),
                   hp.Int('cnn_%d_kernel' % i, 3, 7, step=2),
                   activation='relu', padding='same', kernel_initializer=GI(s))(x)
        x = Dropout(hp.Float('cnn_%d_dropout' % i, 0.0, 0.3, step=0.1), seed=s)(x)
        if i < 2 and int(x.shape[1]) > 2: x = MaxPooling1D(2, padding='same')(x)
    return x

def _lstm(x, hp, s):
    nl = hp.Int('num_lstm_layers', 1, 3)
    for i in range(nl):
        x = LSTM(hp.Int('lstm_%d_units' % i, **UNIT), return_sequences=(i < nl-1),
                 dropout=hp.Float('lstm_%d_dropout' % i, 0.0, 0.3, step=0.1),
                 recurrent_dropout=hp.Float('lstm_%d_rdropout' % i, 0.0, 0.3, step=0.1),
                 kernel_initializer=GI(s))(x)
    return x

def build_model(hp, arch, seed=SEED):
    inp = Input(shape=(WINDOW, 1))
    if arch == 'sequential':
        merged = _lstm(_cnn(inp, hp, seed), hp, seed)
    else:
        c = _cnn(inp, hp, seed)
        c = (GlobalMaxPooling1D() if hp.Choice('cnn_pooling', ['max','avg']) == 'max'
             else GlobalAveragePooling1D())(c)
        merged = Concatenate()([c, _lstm(inp, hp, seed)])
    x = merged
    for i in range(hp.Int('num_dense_layers', 1, 3)):
        x = Dense(hp.Int('dense_%d_units' % i, 8, 256, step=8), activation='relu',
                  kernel_initializer=GI(seed))(x)
        x = Dropout(hp.Float('dense_%d_dropout' % i, 0.0, 0.5, step=0.1), seed=seed)(x)
    out = Dense(1, activation='linear', kernel_initializer=GI(seed))(x)
    lr  = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4, 1e-5])
    opt = {'adam': tf.keras.optimizers.Adam, 'rmsprop': tf.keras.optimizers.RMSprop,
           'sgd': tf.keras.optimizers.SGD}[hp.Choice('optimizer', ['adam','rmsprop','sgd'])](lr)
    m = Model(inp, out); m.compile(optimizer=opt, loss='mse', metrics=['mae']); return m

In [ ]:
def rmse_onestep(m, d, part):
    p = m.predict(d['X'+part], batch_size=512, verbose=0).ravel()
    return float(np.sqrt(mean_squared_error(d['p'+part] + d['y'+part], d['p'+part] + p)))

def rollout(m, start, n):
    win, out = list(S_ALL[start-WINDOW:start]), []
    for _ in range(n):
        x = np.asarray(win[-WINDOW:], np.float32).reshape(1, WINDOW, 1)
        lvl = float(np.clip(win[-1] + float(m(x, training=False).numpy().ravel()[0]), 0.0, 1.2))
        out.append(lvl); win.append(lvl)
    return np.asarray(out, np.float32)

def rmse_recursive(m, d, part):
    st, n = (d['val_start'], len(d['Xv'])) if part == 'v' else (d['test_start'], len(HI)-d['test_start'])
    return float(np.sqrt(mean_squared_error(S_ALL[st:st+n], rollout(m, st, n))))

def score(m, d, protocol, part):
    return (rmse_onestep if protocol == 'onestep' else rmse_recursive)(m, d, part)

def persistence(d, protocol):
    if protocol == 'onestep':
        return float(np.sqrt(mean_squared_error(d['pt'] + d['yt'], d['pt'])))
    st = d['test_start']; n = len(HI) - st
    return float(np.sqrt(mean_squared_error(S_ALL[st:], np.full(n, S_ALL[st-1]))))

def train_one(m, d, ep, pat):
    m.fit(d['Xf'], d['yf'], validation_data=(d['Xv'], d['yv']), epochs=ep, batch_size=BATCH,
          verbose=0, shuffle=True,
          callbacks=[EarlyStopping('val_loss', patience=pat, restore_best_weights=True),
                     ReduceLROnPlateau('val_loss', factor=0.5, patience=8, min_lr=1e-7)])
    return m

OBJ = 'val_rmse_mean'
class MultiSeedBO(kt.BayesianOptimization):
    def __init__(self, arch, data, protocol, **kw):
        self.arch, self.data, self.protocol = arch, data, protocol; super().__init__(**kw)
    def run_trial(self, trial, *a, **kw):
        hp, sc = trial.hyperparameters, []
        for s in range(SEEDS_TRIAL):
            K.clear_session(); gc.collect()
            np.random.seed(SEED+s); tf.random.set_seed(SEED+s)
            try:
                m = train_one(build_model(hp, self.arch, SEED+s), self.data,
                              SEARCH_EPOCHS, SEARCH_PATIENCE)
                v = score(m, self.data, self.protocol, 'v')
                if not np.isfinite(v): v = 1e3
            except Exception as e:
                print('   seed %d failed (%s)' % (s, type(e).__name__)); v = 1e3
            sc.append(v)
        print('   trial %s: %.5f (std %.5f)' % (trial.trial_id, np.mean(sc), np.std(sc, ddof=1)))
        return {OBJ: float(np.mean(sc))}

## Run the searches

In [ ]:
def run_search(arch, protocol):
    split = SPLIT_ONESTEP if protocol == 'onestep' else SPLIT_RECUR
    d, tag = DATA[split], '%s_%s' % (arch, protocol)
    print('\n' + '='*72); print('%s | split %d | %d trials x %d seeds'
                                % (tag.upper(), split, MAX_TRIALS, SEEDS_TRIAL)); print('='*72)
    t0 = time.time()
    tuner = MultiSeedBO(arch=arch, data=d, protocol=protocol,
                        hypermodel=lambda hp: build_model(hp, arch),
                        objective=kt.Objective(OBJ, direction='min'),
                        max_trials=MAX_TRIALS, seed=SEED,
                        directory='bo_battery_b0005', project_name=tag, overwrite=False)
    tuner.search()
    hp = tuner.get_best_hyperparameters(1)[0]
    # retrain the selected configuration on seeds never seen during the search
    sc, params = [], None
    for s in range(SEEDS_FINAL):
        K.clear_session(); gc.collect()
        sd = SEED + 1000 + s; np.random.seed(sd); tf.random.set_seed(sd)
        m = build_model(hp, arch, sd); params = m.count_params()
        train_one(m, d, FINAL_EPOCHS, FINAL_PATIENCE)
        sc.append(score(m, d, protocol, 't'))
    r = dict(arch=arch, protocol=protocol, split=split, params=int(params),
             best_hp=dict(hp.values), per_seed=[round(v,5) for v in sc],
             mean=float(np.mean(sc)), std=float(np.std(sc, ddof=1)),
             best=float(np.min(sc)), persistence=persistence(d, protocol),
             seconds=round(time.time()-t0))
    print('\n%s -> %.5f +/- %.5f SOH  (%.4f Ah)  best seed %.5f  | %.0f s'
          % (tag, r['mean'], r['std'], r['mean']*C0, r['best'], r['seconds']))
    return tuner, r

RESULTS, TUNERS = {}, {}
for arch in ('sequential', 'parallel'):
    for protocol in ('onestep', 'recursive'):
        k = '%s_%s' % (arch, protocol)
        TUNERS[k], RESULTS[k] = run_search(arch, protocol)

In [ ]:
res = pd.DataFrame([dict(architecture=r['arch'], protocol=r['protocol'],
                         params=r['params'], rmse_SOH=round(r['mean'], 5),
                         std=round(r['std'], 5), rmse_Ah=round(r['mean']*C0, 4),
                         best_seed=round(r['best'], 5),
                         persistence=round(r['persistence'], 5))
                    for r in RESULTS.values()])
pd.set_option('display.width', 200)
print('Bayesian optimization results — B0005\n'); print(res.to_string(index=False))
print('\nSelected configurations')
for k, r in RESULTS.items():
    print('\n--- %s (%s params) ---' % (k, format(r['params'], ',')))
    for kk, vv in sorted(r['best_hp'].items()): print('   %-22s %s' % (kk, vv))

## Benchmark table

Published values are on the normalised SOH scale (0-1), so ours are converted from Ah by
dividing by the initial capacity of 1.8565 Ah. Tian and Gao both forecast recursively from a
start point, so the **recursive** rows are the ones that compare like with like; the one-step
rows are included for continuity with the current draft and are marked as a different task.

In [ ]:
PUBLISHED = [
    ('CNN-BiLSTM-AM',      'Tian et al., Batteries 2022 [53]',        0.00481, 'recursive'),
    ('1D CNN-BiLSTM',      'Gao et al., Meas. & Control 2023',        0.00772, 'recursive'),
    ('CNN-BiLSTM',         'Tian et al. [53]',                        0.00595, 'recursive'),
    ('BiLSTM',             'Tian et al. [53]',                        0.00620, 'recursive'),
    ('1D CNN',             'Gao et al.',                              0.00866, 'recursive'),
    ('CNN',                'Tian et al. [53]',                        0.00863, 'recursive'),
    ('1D CNN-LSTM',        'Gao et al.',                              0.00932, 'recursive'),
    ('1D CNN-RNN',         'Gao et al.',                              0.01661, 'recursive'),
    ('RNN',                'Gao et al.',                              0.01776, 'recursive'),
    ('BiLSTM',             'Gao et al.',                              0.01986, 'recursive'),
    ('LSTM',               'Gao et al.',                              0.02302, 'recursive'),
    ('CL-Net',             'Khan et al., Mathematics 2021',           0.205,   'not stated'),
    ('Encoder-decoder',    'Khan et al.',                             0.221,   'not stated'),
    ('CNN-LSTM',           'Khan et al.',                             0.247,   'not stated'),
    ('LSTM',               'Khan et al.',                             0.259,   'not stated'),
]
OURS_PUBLISHED = [
    ('Ours: Parallel CNN-LSTM (as published)',      0.047/1.8565, 'one-step'),
    ('Ours: Sequential + Hyperband (as published)', 0.094/1.8565, 'one-step'),
    ('Ours: Sequential (as published)',             0.350/1.8565, 'one-step'),
    ('Ours: Sequential + GA (as published)',        0.403/1.8565, 'one-step'),
]
rows = [dict(method=m, source=s, rmse_SOH=round(v, 5), protocol=p) for m, s, v, p in PUBLISHED]
rows += [dict(method=m, source='this work', rmse_SOH=round(v, 5), protocol=p)
         for m, v, p in OURS_PUBLISHED]
for k, r in RESULTS.items():
    rows.append(dict(method='Ours: %s CNN-LSTM + Bayesian' % r['arch'].capitalize(),
                     source='this work (new)', rmse_SOH=round(r['mean'], 5),
                     protocol=r['protocol']))
    rows.append(dict(method='   persistence floor (%s)' % r['protocol'], source='reference',
                     rmse_SOH=round(r['persistence'], 5), protocol=r['protocol']))

bench = pd.DataFrame(rows).drop_duplicates(subset=['method','source','protocol'])
for pr in ('recursive', 'one-step'):
    sub = bench[bench.protocol == pr].sort_values('rmse_SOH').reset_index(drop=True)
    if len(sub):
        print('\n' + '='*90); print('PROTOCOL: %s' % pr.upper()); print('='*90)
        print(sub.to_string(index=False))
oth = bench[~bench.protocol.isin(['recursive','one-step'])].sort_values('rmse_SOH')
if len(oth):
    print('\nProtocol not stated in source'); print(oth.to_string(index=False))
bench.sort_values(['protocol','rmse_SOH']).to_csv('benchmark_battery_b0005_with_bayesian.csv', index=False)

In [ ]:
out = dict(dataset='NASA Battery B0005', C0_Ah=float(C0), window=WINDOW,
           max_trials=MAX_TRIALS, seeds_per_trial=SEEDS_TRIAL, seeds_final=SEEDS_FINAL,
           searches=RESULTS)
json.dump(out, open('bayesian_battery_b0005_results.json', 'w'), indent=2)
res.to_csv('bayesian_battery_b0005_summary.csv', index=False)
print('saved bayesian_battery_b0005_results.json, summary csv and benchmark csv')